# Phase 3: Email Spam Detection Modeling
This notebook trains and evaluates 5 Machine Learning models on the Email Spam Dataset:
1. Logistic Regression
2. Multinomial Naive Bayes
3. Linear SVM
4. XGBoost
5. Random Forest

We will preprocess raw email bodies using our custom `EmailPreprocessor` utility and TF-IDF vectorization, evaluate the models, and persist the best classifier.

In [26]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import sys
sys.path.append('..')
from app.utils.email_preprocessor import EmailPreprocessor

sns.set_theme(style="whitegrid")

## 1. Load and Vectorize Dataset
We load the email dataset containing raw message text, convert the labels `ham -> 0, spam -> 1`, split the dataset, and apply text cleaning + TF-IDF vectorization.

In [27]:
email_path = '../datasets/spamassassin/email_spam_dataset.csv'
df = pd.read_csv(email_path)
df = df[['label', 'text']].dropna()
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

X = df['text']
y = df['label']

print(f"Email dataset class distribution:\n{y.value_counts(normalize=True)*100}")

Email dataset class distribution:
label
0    86.593683
1    13.406317
Name: proportion, dtype: float64


In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocessor = EmailPreprocessor(max_features=3000)
X_train_vectorized = preprocessor.fit_transform(X_train)
X_test_vectorized = preprocessor.transform(X_test)

# Save the TF-IDF Vectorizer
preprocessor.save_vectorizer("../models/tfidf_vectorizer.joblib")
print(f"Saved TF-IDF vectorizer. Matrix shape: {X_train_vectorized.shape}")

Saved TF-IDF vectorizer to ../models/tfidf_vectorizer.joblib
Saved TF-IDF vectorizer. Matrix shape: (4457, 3000)


## 2. Train and Evaluate Models

In [29]:
def evaluate_model(name, clf, X_tr, X_te, y_tr, y_te):
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    train_time = time.time() - t0
    
    t1 = time.time()
    preds = clf.predict(X_te)
    inf_latency = (time.time() - t1) / X_te.shape[0] * 1000
    
    return {
        "accuracy": accuracy_score(y_te, preds),
        "precision": precision_score(y_te, preds, zero_division=0),
        "recall": recall_score(y_te, preds, zero_division=0),
        "f1_score": f1_score(y_te, preds, zero_division=0),
        "train_time_sec": train_time,
        "inference_latency_ms": inf_latency
    }

In [30]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": SVC(kernel='linear', probability=True, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results = {}
best_f1 = -1
best_model_name = ""
best_model_obj = None

for name, clf in models.items():
    print(f"Evaluating {name}...")
    metrics = evaluate_model(name, clf, X_train_vectorized, X_test_vectorized, y_train, y_test)
    results[name] = metrics
    print(f"  F1-Score: {metrics['f1_score']:.4f} | Accuracy: {metrics['accuracy']:.4f}")
    
    is_better = False
    f1_rounded = round(metrics['f1_score'], 5)
    best_f1_rounded = round(best_f1, 5)

    if f1_rounded > best_f1_rounded:
        is_better = True
    elif f1_rounded == best_f1_rounded:
        if best_model_obj is None or metrics['inference_latency_ms'] < results[best_model_name]['inference_latency_ms']:
            is_better = True

    if is_better:
        best_f1 = metrics['f1_score']
        best_model_name = name
        best_model_obj = clf


Evaluating Logistic Regression...
  F1-Score: 0.8889 | Accuracy: 0.9731
Evaluating Multinomial Naive Bayes...


  F1-Score: 0.8971 | Accuracy: 0.9749
Evaluating Linear SVM...
  F1-Score: 0.9433 | Accuracy: 0.9857
Evaluating XGBoost...
  F1-Score: 0.9007 | Accuracy: 0.9749
Evaluating Random Forest...
  F1-Score: 0.8963 | Accuracy: 0.9749


## 3. Save Best Email Classifier & Compile Metrics

In [31]:
print(f"Saving best model ({best_model_name}) to models/email_best_model.joblib...")
joblib.dump(best_model_obj, "../models/email_best_model.joblib")

# Read URL metrics from intermediate file to build comparison_metrics.json
url_metrics = {}
best_url = "XGBoost"

if os.path.exists("../models/url_metrics.json"):
    with open("../models/url_metrics.json", "r") as f:
        url_metrics = json.load(f)
    if url_metrics:
        # Choose best URL model using F1-score and inference latency as tiebreaker
        valid_keys = [k for k in url_metrics.keys() if '(Full)' not in k]
        best_url = max(valid_keys, key=lambda k: (round(url_metrics[k]['f1_score'], 5), -url_metrics[k]['inference_latency_ms']))

comparison_data = {
    "url_models": url_metrics,
    "email_models": results,
    "best_url_model": best_url,
    "best_email_model": best_model_name
}

with open("../models/comparison_metrics.json", "w") as f:
    json.dump(comparison_data, f, indent=4)



Saving best model (Linear SVM) to models/email_best_model.joblib...
